In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# ==========================================
# CONFIGURATION
# ==========================================
DATASET_PATH = r"C:\Users\ADMIN\OneDrive\Desktop\NLP\NLP\data\dataset.jsonl" 
OUTPUT_DIR = "./eda_results"
# ==========================================

def extract_entities(tags):
    entities = []
    current_entity = None
    length = 0
    
    for tag in tags:
        if tag.startswith('B-'):
            if current_entity:
                entities.append({'entity_type': current_entity, 'length': length})
            current_entity = tag[2:]
            length = 1
        elif tag.startswith('I-') and current_entity == tag[2:]:
            length += 1
        else:
            if current_entity:
                entities.append({'entity_type': current_entity, 'length': length})
                current_entity = None
                length = 0
                
    if current_entity:
        entities.append({'entity_type': current_entity, 'length': length})
        
    return entities

def check_iob_format(tags):
    errors = 0
    prev_tag = 'O'
    for tag in tags:
        if tag.startswith('I-'):
            entity_type = tag[2:]
            if prev_tag == 'O' or (prev_tag != 'O' and prev_tag[2:] != entity_type):
                errors += 1
        prev_tag = tag
    return errors

def main():
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"Dataset missing: {DATASET_PATH}")
        
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Load Data
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    df = dataset.train_test_split(test_size=0.2, seed=42)["train"].to_pandas()

    # 2. Extract Unique Labels
    all_tags = [tag for sublist in df['ner_tags'] for tag in sublist]
    unique_tags = sorted(list(set(all_tags)))
    
    with open(os.path.join(OUTPUT_DIR, "unique_labels.txt"), "w") as f:
        f.write("\n".join(unique_tags))
    
    print(f"Found {len(unique_tags)} unique labels: {unique_tags}")

    # 3. Sequence Length Distribution
    df['seq_length'] = df['tokens'].apply(len)
    
    plt.figure(figsize=(10, 5))
    sns.histplot(df['seq_length'], bins=20, kde=True, color='#2c7fb8')
    plt.title("Sequence Length Distribution")
    plt.xlabel("Tokens")
    plt.ylabel("Frequency")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(OUTPUT_DIR, "seq_length_dist.png"))
    plt.close()

    print(f"Seq Length -> Max: {df['seq_length'].max()} | Mean: {df['seq_length'].mean():.2f}")

    # 4. Entity Distribution & Length
    df['entities'] = df['ner_tags'].apply(extract_entities)
    df_entities = pd.DataFrame([ent for sublist in df['entities'] for ent in sublist])

    if not df_entities.empty:
        # Save Entity Stats
        entity_stats = df_entities['entity_type'].value_counts().reset_index()
        entity_stats.columns = ['Entity_Type', 'Count']
        entity_stats.to_csv(os.path.join(OUTPUT_DIR, "entity_distribution.csv"), index=False)

        # Plot Entity Distribution
        plt.figure(figsize=(10, 5))
        sns.countplot(data=df_entities, y='entity_type', order=entity_stats['Entity_Type'], palette="viridis")
        plt.title("Entity Count")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "entity_count.png"))
        plt.close()

        # Plot Entity Lengths
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df_entities, x='length', y='entity_type', palette="Set2")
        plt.title("Entity Lengths")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "entity_lengths.png"))
        plt.close()
        
    # 5. Data Quality Validation
    total_errors = df['ner_tags'].apply(check_iob_format).sum()
    print(f"IOB Format Errors: {total_errors}")

if __name__ == "__main__":
    main()

Found 11 unique labels: ['B-EXP', 'B-LOC', 'B-ROLE', 'B-SALARY', 'B-SKILL', 'I-EXP', 'I-LOC', 'I-ROLE', 'I-SALARY', 'I-SKILL', 'O']
Seq Length -> Max: 103 | Mean: 21.63


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2100\2866829139.py:94: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df_entities, y='entity_type', order=entity_stats['Entity_Type'], palette="viridis")
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2100\2866829139.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_entities, x='length', y='entity_type', palette="Set2")


IOB Format Errors: 0
